In [12]:
import pandas as pd
import numpy as np
import time

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, roc_auc_score, precision_score, recall_score, f1_score

from imblearn.over_sampling import SMOTE
from xgboost import XGBClassifier

In [13]:
df = pd.read_csv('../Datasets/preprocessed_liver_data.csv')

X = df.drop('Result', axis=1)
y = df['Result']

In [14]:
skf = StratifiedKFold(
    n_splits=10,
    shuffle=True,
    random_state=42
)

xgb_acc = []
xgb_auc = []
xgb_prec = []
xgb_rec = []
xgb_f1 = []

In [15]:
train_times = []
predict_times = []
for fold, (train_idx, val_idx) in enumerate(skf.split(X, y), 1):
    print(f"\n--- XGBoost Fold {fold} ---")

    X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]

    # SMOTE only on training fold
    smote = SMOTE(random_state=42)
    X_train_bal, y_train_bal = smote.fit_resample(X_train, y_train)

    # XGBoost model
    xgb = XGBClassifier(
        n_estimators=200,
        max_depth=5,
        learning_rate=0.1,
        subsample=0.8,
        colsample_bytree=0.8,
        eval_metric='logloss',
        random_state=42,
        n_jobs=-1
    )

   # -------- Training time --------
    start_train = time.perf_counter()
    xgb.fit(X_train_bal, y_train_bal)
    end_train = time.perf_counter()

    train_times.append(end_train - start_train)

    # -------- Prediction time --------
    start_pred = time.perf_counter()

    y_pred = xgb.predict(X_val)
    y_prob = xgb.predict_proba(X_val)[:,1]

    end_pred = time.perf_counter()

    predict_times.append(end_pred - start_pred)

    # Predictions
    y_pred = xgb.predict(X_val)
    y_prob = xgb.predict_proba(X_val)[:, 1]
    
    # Metrics
    acc = accuracy_score(y_val, y_pred)
    auc = roc_auc_score(y_val, y_prob)
    prec = precision_score(y_val, y_pred)
    rec = recall_score(y_val, y_pred)
    f1 = f1_score(y_val, y_pred)
    
    xgb_acc.append(acc)
    xgb_auc.append(auc)
    xgb_prec.append(prec)
    xgb_rec.append(rec)
    xgb_f1.append(f1)
    
    print(f"Accuracy  : {acc:.4f}")
    print(f"ROC-AUC   : {auc:.4f}")
    print(f"Precision : {prec:.4f}")
    print(f"Recall    : {rec:.4f}")
    print(f"F1-score  : {f1:.4f}")


--- XGBoost Fold 1 ---
Accuracy  : 0.9954
ROC-AUC   : 0.9999
Precision : 0.9993
Recall    : 0.9942
F1-score  : 0.9967

--- XGBoost Fold 2 ---
Accuracy  : 0.9933
ROC-AUC   : 0.9997
Precision : 0.9964
Recall    : 0.9942
F1-score  : 0.9953

--- XGBoost Fold 3 ---
Accuracy  : 0.9959
ROC-AUC   : 0.9997
Precision : 0.9985
Recall    : 0.9957
F1-score  : 0.9971

--- XGBoost Fold 4 ---
Accuracy  : 0.9933
ROC-AUC   : 0.9998
Precision : 0.9964
Recall    : 0.9942
F1-score  : 0.9953

--- XGBoost Fold 5 ---
Accuracy  : 0.9917
ROC-AUC   : 0.9983
Precision : 0.9956
Recall    : 0.9928
F1-score  : 0.9942

--- XGBoost Fold 6 ---
Accuracy  : 0.9979
ROC-AUC   : 1.0000
Precision : 0.9986
Recall    : 0.9986
F1-score  : 0.9986

--- XGBoost Fold 7 ---
Accuracy  : 0.9933
ROC-AUC   : 0.9984
Precision : 0.9949
Recall    : 0.9957
F1-score  : 0.9953

--- XGBoost Fold 8 ---
Accuracy  : 0.9912
ROC-AUC   : 0.9995
Precision : 0.9928
Recall    : 0.9949
F1-score  : 0.9939

--- XGBoost Fold 9 ---
Accuracy  : 0.9943
ROC-A

In [16]:
print("\n====== XGBOOST (CV RESULTS) ======")

print(f"Accuracy  : {np.mean(xgb_acc):.4f} ± {np.std(xgb_acc):.4f}")
print(f"ROC-AUC   : {np.mean(xgb_auc):.4f} ± {np.std(xgb_auc):.4f}")
print(f"Precision : {np.mean(xgb_prec):.4f}")
print(f"Recall    : {np.mean(xgb_rec):.4f}")
print(f"F1-score  : {np.mean(xgb_f1):.4f}")

print("\n====== XGBoost Timing ======")

print(f"Average Training Time  : {np.mean(train_times):.4f} seconds")
print(f"Average Prediction Time: {np.mean(predict_times):.6f} seconds")


====== XGBOOST (CV RESULTS) ======
Accuracy  : 0.9940 ± 0.0019
ROC-AUC   : 0.9994 ± 0.0006
Precision : 0.9970
Recall    : 0.9946
F1-score  : 0.9958

====== XGBoost Timing ======
Average Training Time  : 0.5844 seconds
Average Prediction Time: 0.013389 seconds
